(course-core-discovery-and-attributes)=

# Module 10: Discovery & Attributes

In MolSysMT, you don't need to guess what an object can do. Because of our form-agnostic philosophy, some "Forms" might contain more information than others. 

For example, an **H5MSM** file usually contains everything (topology, coordinates, bonds, boxes). But a simple **PDB** file might be missing box information, and an **XTC** file contains only coordinates and boxes, but no atom names.

In this module, you will learn how to audit your systems and understand the concept of **Attributes**. Contractual chemical forms such as RDKit, OpenFF, ParmEd, SMILES/SMI, MOL2, and PSF expose different attribute sets by design: a SMILES graph has no coordinates or partial charges, while a conformer-bearing OpenFF Molecule can have both. A PSF carries force-field atom types, partial charges, and connectivity, but not coordinates or chemical bond orders.

### 1. What are Attributes?

An **Attribute** is any property of a molecular system that defines its identity, structure, or state. In MolSysMT, attributes are the data points that you can both query (using `get`) and modify (using `set`).

Attributes are classified into several non-exclusive semantic layers. For example, `atom_index` and `n_atoms` are both topological and structural because they identify the stable atom inventory and the atom axis of structural arrays. They are not chemical-state attributes: chemical states align against that inventory, but resolving another state does not change it. The principal layers are:

1.  **Stable Topological Attributes:** Data that defines atom identity and semantic hierarchy, such as atom names, group IDs, molecules, and chains.
2.  **Structural Attributes:** Geometric data that describes the state of the system, like coordinates, velocities, and simulation box dimensions.
3.  **Chemical-State Attributes:** Bonds, covalent components, formal charge, aromaticity, radicals, implicit hydrogens, and stereochemistry for a resolved chemical state. Rich bonds keep formal and fractional order, relationship kind, aromaticity, conjugation, stereochemistry and reference atoms, direction, component participation, and evidence independent. `get()`, `set()`, `has_attribute()`, and `select()` accept `chemical_state='reference'`, `chemical_state='structure'`, or a 0-based integer index. `structure_chemical_state_index` associates MolSys frames with states; a multi-frame request must resolve one known state. The index is scoped to the call and state IDs remain non-unique labels, not selectors. Stable `isotope` is an atom attribute rather than a chemical-state field.
4.  **Conversion Fidelity:** `convert(..., return_report=True)` returns an immutable `ConversionReport` with `exact`, `equivalent`, or `lossy` outcome, explicit `audited_scopes`, an `is_exhaustive` flag, and structured issues. Every `ConversionIssue` names the affected `attribute`, issue `kind`, semantic `scope`, and explanatory `reason`. `strict=True` rejects detected lossy conversions before target creation.
5.  **Mechanical and Physical Attributes:** Force-field parameters or calculated quantities such as partial charge, atom force-field type, mass, and potential energy.

> **Deep Dive:** For a complete list of all molecular system properties and their formal definitions, visit the [**Attributes Section**](https://www.uibcdf.org/molsysmt) in our User Guide Foundations.

In [1]:
import molsysmt as msm
from molsysmt import systems

lysozyme = systems['T4 lysozyme L99A']['181l.bcif.gz']

### 2. Checking for specific information
The function `has_attribute()` tells you if a specific property is present in your object.

In [2]:
print(f"Has coordinates? {msm.has_attribute(lysozyme, 'coordinates')}")
print(f"Has box? {msm.has_attribute(lysozyme, 'box')}")
print(f"Has velocities? {msm.has_attribute(lysozyme, 'velocities')}")

Has coordinates? True
Has box? True
Has velocities? True


### 3. Listing all available attributes
You can get a complete list of all properties your object currently "knows" using `get_attributes()`.

In [3]:
available_atts = msm.get_attributes(lysozyme)
print(f"This system has {len(available_atts)} different attributes.")
print(f"Examples: {available_atts[:10]}")

This system has 65 different attributes.
Examples: ['atom_index', 'atom_name', 'atom_id', 'atom_type', 'group_index', 'group_name', 'group_id', 'group_type', 'component_index', 'component_name']


### 4. Understanding Form limitations
Let's compare a full system with a trajectory-only file. We'll take a DCD file from the database.

In [4]:
dcd_file = systems['chicken villin HP35']['traj_chicken_villin_HP35_solvated.dcd']

print(f"DCD Form: {msm.get_form(dcd_file)}")
print(f"DCD has atom names? {msm.has_attribute(dcd_file, 'atom_name')}")
print(f"DCD has coordinates? {msm.has_attribute(dcd_file, 'coordinates')}")

DCD Form: file:dcd
DCD has atom names? False
DCD has coordinates? True


### 5. Where is the data?
When you work with multiple files combined, `where_is_attribute()` is your best friend to debug which form is providing each specific property.

In [5]:
topology = systems['chicken villin HP35']['chicken_villin_HP35_solvated.h5msm']
trajectory = systems['chicken villin HP35']['traj_chicken_villin_HP35_solvated.dcd']

location = msm.where_is_attribute([topology, trajectory], 'atom_name')
print(f"Atom names are being fetched from: {location}")

Atom names are being fetched from: ('/home/diego/repos@uibcdf/molsysmt/molsysmt/data/h5msm/chicken_villin_HP35_solvated.h5msm', 'file:h5msm')


--- 

### 🏆 Challenge 7: The Data Auditor

1. Load a system from a PDB ID (e.g., `'pdb_id:181L'`).
2. Use `msm.get_attributes()` to see if it contains **Bonds** information.
3. Check if it contains **Occupancy** or **B-factors** (these are structural attributes common in PDBs).
4. Count how many total attributes this system has compared to a local H5MSM file.

Now you can audit any file you receive! In **Module 3**, we will learn how to combine different files to create a complete system.